In [76]:
import numpy as np, pandas as pd, requests, re, time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException, StaleElementReferenceException
from selenium.webdriver.common.keys import Keys


In [77]:
chrome_options = Options()
chrome_options.add_experimental_option("detach", True)

In [78]:
url = 'https://www.imdb.com/fr/title/tt27497448/reviews/'

driver = webdriver.Chrome(options=chrome_options)
driver.get(url)


In [79]:
# Cliquer sur le bouton "Accepter les cookies"
time.sleep(5)
driver.find_element(By.CSS_SELECTOR, "[data-testid='accept-button']").click()

In [80]:
# Cliquer sur le bouton "Afficher tout" pour voir tous les commentaires
time.sleep(2)

while True :
    try:
        afficherTout = driver.find_element(By.CLASS_NAME, "ipc-see-more__text")
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'})", afficherTout)
        time.sleep(1)
        afficherTout.click()
    except NoSuchElementException:
        break
    except ElementClickInterceptedException:
        afficherTout = driver.find_element(By.CLASS_NAME, "ipc-see-more__text")
        driver.execute_script("arguments[0].click();", afficherTout)
        time.sleep(1)
    except StaleElementReferenceException:
        time.sleep(2)
        try:
            afficherTout = driver.find_element(By.CLASS_NAME, "ipc-see-more__text")
            afficherTout.click()
        except NoSuchElementException:
            break

        




In [81]:
# Récupérer les données d'un coup
# Il faut tout faire d'un coup car parfois il n'y a pas de notes ou pas de commentaires
# Et donc un décalage qui rendrait l'analyse caduque.

notes = []
titres = []
commentaires = []
dates = []

time.sleep(10)

reviews = driver.find_elements(By.CLASS_NAME, "user-review-item")

for review in reviews:
    try:
        note = review.find_element(By.CLASS_NAME, "ipc-rating-star--rating")
        notes.append(note.text)

    except NoSuchElementException:
        notes.append(None)

    try:
        titre = review.find_element(By.CLASS_NAME, "ipc-title__text")
        titres.append(titre.text)
    except NoSuchElementException:
        titres.append(None)

    try:
        commentaire = review.find_element(By.CLASS_NAME, "ipc-html-content-inner-div")
        commentaires.append(commentaire.text)
    except NoSuchElementException:
        commentaires.append(None)

    try:
        date = review.find_element(By.CLASS_NAME, "review-date")
        dates.append(date.text)

    except NoSuchElementException:
        dates.append(None)


print(len(notes))
print(len(titres))
print(len(commentaires))
print(len(dates))

print(notes)
print(titres)
print(commentaires)
print(dates)

1262
1262
1262
1262
['8', '9', '9', '9', '9', '8', '10', '10', '8', '8', '6', '10', '10', '10', '9', '6', '8', '8', None, '10', '10', '6', '10', '10', '9', '7', '9', '6', '3', None, '10', '10', '9', '10', '6', '6', '10', '10', '8', '10', '10', '9', '6', '10', '6', '3', None, '10', '10', '9', '10', '8', '10', '7', '9', '6', '3', '10', '10', '3', '10', '10', '8', '6', '10', '6', '2', '9', '9', '9', '10', '10', '6', '9', '9', '2', '10', '10', '10', '9', '10', '7', '10', '6', '5', '9', '9', '10', '9', '10', '10', '7', '6', '2', '8', '10', '10', '9', '10', '10', '7', '10', '7', '4', '9', '9', '10', '10', '10', '7', '8', '7', '2', '9', '10', '10', '10', '9', '8', None, '7', '10', '10', '9', '10', '9', '9', '4', '7', '2', '2', '2', '2', '4', '1', '3', '3', '2', '1', '1', '1', '1', '3', '1', '1', '1', '4', '4', '3', '1', '5', '3', '1', '1', '3', '2', '3', '2', '1', '6', '1', '3', '1', '1', '2', '7', '5', '2', '1', '3', '3', '5', '3', '2', '1', '4', '1', '9', '9', '9', '9', '8', '9', '9', '9', 

In [82]:
# Mettre les données dans un dataframe

columns = ["notesImdb", "titres", "avis", "dates"]

df_knight = pd.DataFrame(list(zip(notes, titres, commentaires, dates)), columns=columns)

df_knight["series"] = "A Knight of the Seven Kingdoms"
df_knight

,notesImdb,titres,avis,dates,series
0,8,Honest Opinion,Turns out Westeros is way more interesting whe...,24 janv. 2026,A Knight of the Seven Kingdoms
1,9,"So Far, So Good (Don't Screw It Up)",The first episode of The Knight of the Seven K...,19 janv. 2026,A Knight of the Seven Kingdoms
2,9,A proper return to Westeros,"I stayed up for the 3 am UK premiere and, afte...",19 janv. 2026,A Knight of the Seven Kingdoms
3,9,The only disappointment i had with ep 1,Its only 42 minuts. I expected and hour to be ...,19 janv. 2026,A Knight of the Seven Kingdoms
4,9,Really freaking good with only one bad moment ...,"A Knight of the Seven Kingdoms, based on the f...",19 janv. 2026,A Knight of the Seven Kingdoms
...,...,...,...,...,...
1257,None,Amazing,The casting of characters in A Knight of the S...,23 févr. 2026,A Knight of the Seven Kingdoms
1258,10,A Beauty of Westeros,I have a close friend who has read the books a...,23 févr. 2026,A Knight of the Seven Kingdoms
1259,4,Old stories,I feel like this is not a series but a long mo...,23 févr. 2026,A Knight of the Seven Kingdoms
1260,1,Very bad show,I don't understand the message?? This is show ...,20 févr. 2026,A Knight of the Seven Kingdoms


### Série 2 : King & Conquerer

In [86]:
# Mettre la série dans la bare de recherche
researchBox = driver.find_element(By.CLASS_NAME, "imdb-header-search__input")
researchBox.clear()
researchBox.send_keys("King & Conquerer")
researchBox.send_keys(Keys.ENTER)

# Cliquer sur la page
try:
    driver.find_element(By.CLASS_NAME, "ipc-lockup-overlay__screen").click()
except StaleElementReferenceException:
    time.sleep(1)
    driver.find_element(By.CLASS_NAME, "ipc-lockup-overlay__screen").click()

# Cliquer sur les avis
try:
    driver.find_element(By.CLASS_NAME, "less-than-three-Elements").click()
except ElementClickInterceptedException:
    afficherAvis = driver.find_element(By.CLASS_NAME, "less-than-three-Elements")
    driver.execute_script("arguments[0].click();", afficherAvis)
except StaleElementReferenceException:
    time.sleep(1)
    driver.find_element(By.CLASS_NAME, "less-than-three-Elements").click()

In [87]:
# Cliquer sur le bouton "Afficher tout" pour voir tous les commentaires
time.sleep(2)

while True :
    try:
        afficherTout = driver.find_element(By.CLASS_NAME, "ipc-see-more__text")
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'})", afficherTout)
        time.sleep(1)
        afficherTout.click()
    except NoSuchElementException:
        break
    except ElementClickInterceptedException:
        afficherTout = driver.find_element(By.CLASS_NAME, "ipc-see-more__text")
        driver.execute_script("arguments[0].click();", afficherTout)
        time.sleep(1)
    except StaleElementReferenceException:
        time.sleep(2)
        try:
            afficherTout = driver.find_element(By.CLASS_NAME, "ipc-see-more__text")
            afficherTout.click()
        except NoSuchElementException:
            break

In [88]:
# Récupérer les données d'un coup
# Il faut tout faire d'un coup car parfois il n'y a pas de notes ou pas de commentaires
# Et donc un décalage qui rendrait l'analyse caduque.

notes = []
titres = []
commentaires = []
dates = []

time.sleep(10)

reviews = driver.find_elements(By.CLASS_NAME, "user-review-item")

for review in reviews:
    try:
        note = review.find_element(By.CLASS_NAME, "ipc-rating-star--rating")
        notes.append(note.text)

    except NoSuchElementException:
        notes.append(None)

    try:
        titre = review.find_element(By.CLASS_NAME, "ipc-title__text")
        titres.append(titre.text)
    except NoSuchElementException:
        titres.append(None)

    try:
        commentaire = review.find_element(By.CLASS_NAME, "ipc-html-content-inner-div")
        commentaires.append(commentaire.text)
    except NoSuchElementException:
        commentaires.append(None)

    try:
        date = review.find_element(By.CLASS_NAME, "review-date")
        dates.append(date.text)

    except NoSuchElementException:
        dates.append(None)

print(len(notes))
print(len(titres))
print(len(commentaires))
print(len(dates))

print(notes)
print(titres)
print(commentaires)
print(dates)

178
178
178
178
['5', '5', '5', None, '6', '1', '6', '3', '9', '6', '2', '8', '6', '4', None, '7', '2', '6', '4', '8', '6', '1', '10', '7', '3', None, '7', '4', '6', '1', '10', '7', '2', '8', '6', '1', '7', '1', '7', '1', '8', '7', '2', '10', '7', '1', '6', '2', '6', '1', '8', '6', '3', '8', '7', '3', '1', '1', '9', '2', '8', '4', '2', '2', None, '8', '3', '9', '3', '2', '1', '10', '3', '10', '4', '1', '2', '8', '1', '8', '1', '1', '2', '10', '3', '10', '3', '3', '2', '9', '5', '4', '2', '2', '10', '6', '10', '1', '2', '3', '9', '4', '10', '4', '2', '10', '1', '9', '1', '2', '8', '4', '9', '1', '2', '1', '8', '3', '8', '5', '3', '8', '2', None, '8', '1', '4', '3', '9', '1', '4', '4', '8', '5', '8', '5', '3', '9', '10', '3', '4', '4', '10', '5', '4', '8', '2', '1', '4', '1', '1', '1', '3', '3', '1', '5', '3', '1', '1', '1', '4', '3', '1', '1', '2', '4', '3', '4', '1', '4', '2', '5', '1', '1', '3', '7', '3', '1']
['Average drama, terrible history', 'Why oh why invent history', 'Woeful sc

In [89]:
# Créer le dataframe
df_king = pd.DataFrame(list(zip(notes, titres, commentaires, dates)), columns = columns)

df_king["series"] = "King & Conqueror"
df_king

,notesImdb,titres,avis,dates,series
0,5,"Average drama, terrible history",The cast is packed with talent. The cinematogr...,26 août 2025,King & Conqueror
1,5,Why oh why invent history,"So much money spent, such a brilliant cast, wh...",31 août 2025,King & Conqueror
2,5,Woeful script,I really wanted to love this production & had ...,31 août 2025,King & Conqueror
3,None,Double Standards?,I love historic dramas and was looking forward...,26 août 2025,King & Conqueror
4,6,One of History's Greatest Stories,I used to tell this story to my three sons gro...,1 sept. 2025,King & Conqueror
...,...,...,...,...,...
173,1,Mediocre filmmaking,Weak script in combination with weak direction...,29 oct. 2025,King & Conqueror
174,3,drunk history,"As many other reviewers have noted, the events...",2 févr. 2026,King & Conqueror
175,7,I get it...,None,26 févr. 2026,King & Conqueror
176,3,Realism...,Earl's military crews count 5-8 soldiers... Si...,2 mars 2026,King & Conqueror


In [90]:
# Concaténation et conversion des colonnes
df_imdb = pd.concat([df_knight, df_king], ignore_index=True)

df_imdb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1440 entries, 0 to 1439
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   notesImdb  1412 non-null   object
 1   titres     1440 non-null   object
 2   avis       1238 non-null   object
 3   dates      1440 non-null   object
 4   series     1440 non-null   object
dtypes: object(5)
memory usage: 56.4+ KB


In [91]:
# Je ne que les commenaitres qui ont une note
df_imdb = df_imdb.dropna(subset=["notesImdb"])
df_imdb.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1412 entries, 0 to 1439
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   notesImdb  1412 non-null   object
 1   titres     1412 non-null   object
 2   avis       1217 non-null   object
 3   dates      1412 non-null   object
 4   series     1412 non-null   object
dtypes: object(5)
memory usage: 66.2+ KB


In [92]:
# Changements
df_imdb = df_imdb.astype({'notesImdb' : 'int'})
df_imdb["avis"] = df_imdb['titres'] + " - " + df_imdb['avis'].fillna("")
df_imdb = df_imdb.drop("titres", axis=1)
df_imdb.info()


<class 'pandas.core.frame.DataFrame'>
Index: 1412 entries, 0 to 1439
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   notesImdb  1412 non-null   int64 
 1   avis       1412 non-null   object
 2   dates      1412 non-null   object
 3   series     1412 non-null   object
dtypes: int64(1), object(3)
memory usage: 55.2+ KB


In [93]:
df_imdb["dates"].unique()

array(['24 janv. 2026', '19 janv. 2026', '27 janv. 2026', '20 janv. 2026',
       '22 janv. 2026', '23 janv. 2026', '21 janv. 2026', '26 janv. 2026',
       '25 janv. 2026', '11 févr. 2026', '13 févr. 2026', '12 févr. 2026',
       '10 févr. 2026', '9 févr. 2026', '15 févr. 2026', '8 févr. 2026',
       '14 févr. 2026', '7 févr. 2026', '2 févr. 2026', '16 févr. 2026',
       '19 févr. 2026', '18 févr. 2026', '3 févr. 2026', '5 févr. 2026',
       '30 janv. 2026', '1 févr. 2026', '27 févr. 2026', '23 févr. 2026',
       '20 févr. 2026', '4 févr. 2026', '6 févr. 2026', '17 févr. 2026',
       '24 févr. 2026', '16 mars 2026', '28 janv. 2026', '23 mars 2026',
       '22 mars 2026', '20 mars 2026', '21 févr. 2026', '22 févr. 2026',
       '26 févr. 2026', '4 mars 2026', '7 mars 2026', '1 mars 2026',
       '25 févr. 2026', '2 mars 2026', '19 mars 2026', '29 janv. 2026',
       '10 mars 2026', '28 févr. 2026', '15 mars 2026', '18 mars 2026',
       '3 mars 2026', '31 janv. 2026', '8 mars 202

In [94]:
# Pour la date : traduction anglais pour permettre la conversion.
mois = {
    'janv.': 'January', 'févr.': 'February', 'mars': 'March',
    'avr.': 'April', 'mai': 'May', 'juin': 'June',
    'juil.': 'July', 'août': 'August', 'sept.': 'September',
    'oct.': 'October', 'nov.': 'November', 'déc.': 'December'
}

df_imdb['dates'] = df_imdb['dates'].replace(mois, regex=True)

df_imdb['dates'] = pd.to_datetime(df_imdb['dates'], format='%d %B %Y')

In [95]:
# Vérifications

display(df_imdb)
print() # Saut à la ligne
print(df_imdb.info())

,notesImdb,avis,dates,series
0,8,Honest Opinion - Turns out Westeros is way mor...,2026-01-24,A Knight of the Seven Kingdoms
1,9,"So Far, So Good (Don't Screw It Up) - The firs...",2026-01-19,A Knight of the Seven Kingdoms
2,9,A proper return to Westeros - I stayed up for ...,2026-01-19,A Knight of the Seven Kingdoms
3,9,The only disappointment i had with ep 1 - Its ...,2026-01-19,A Knight of the Seven Kingdoms
4,9,Really freaking good with only one bad moment ...,2026-01-19,A Knight of the Seven Kingdoms
...,...,...,...,...
1435,1,Mediocre filmmaking - Weak script in combinati...,2025-10-29,King & Conqueror
1436,3,drunk history - As many other reviewers have n...,2026-02-02,King & Conqueror
1437,7,I get it... -,2026-02-26,King & Conqueror
1438,3,Realism... - Earl's military crews count 5-8 s...,2026-03-02,King & Conqueror



<class 'pandas.core.frame.DataFrame'>
Index: 1412 entries, 0 to 1439
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   notesImdb  1412 non-null   int64         
 1   avis       1412 non-null   object        
 2   dates      1412 non-null   datetime64[ns]
 3   series     1412 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 55.2+ KB
None


In [96]:
df_imdb.to_csv('df_imdb.csv', index=False)